In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from predict import predict_backbone

/home/monki/Documentos/github/backbone-pas-sat/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import time, lzma, gzip, bz2
from pysat.formula import CNF, IDPool
from pysat.solvers import Solver


def _open_text(path):
    if path.endswith((".xz", ".lzma")): return lzma.open(path, "rt")
    if path.endswith(".gz"): return gzip.open(path, "rt")
    if path.endswith(".bz2"): return bz2.open(path, "rt")
    return open(path, "r")


def load_cnf(path):
    cnf, clause = CNF(), []
    with _open_text(path) as f:
        for line in f:
            line = line.strip()
            if not line or line[0] in "cp%":
                continue
            for tok in line.split():
                lit = int(tok)
                if lit == 0:
                    if clause:
                        cnf.append(clause); clause = []
                else:
                    clause.append(lit)
    if clause:
        cnf.append(clause)
    return cnf

In [3]:
THETA = 0.4
DELTA = 28

In [4]:
cnf_path = "../data/satlib/random_3sat/uf100-01.cnf"
assert os.path.exists(cnf_path), cnf_path
scores = predict_backbone(cnf_path)
scores[1]

0.3973222076892853

In [5]:
S = {var: val for var, val in scores.items() if abs(val - 0.5) >= THETA}
len(S)

28

In [6]:
cnf = load_cnf(cnf_path)

vpool = IDPool(start_from=cnf.nv+1)

In [7]:
with Solver(name='kissat404', bootstrap_with=cnf) as solver:
    sat = solver.solve()
    model = solver.get_model() if sat else None

sat

True

In [8]:
delta_vars = {var: vpool.id() for var in S.keys()}

In [9]:
def create_clause(i: int, p: float, delta_vars: dict) -> list:
    if p - 0.5 <= 0:
        return [i, delta_vars[i]]
    else:
        return [-i, delta_vars[i]]

In [10]:
clauses = [create_clause(i, j, delta_vars=delta_vars) for i, j in S.items() ]
clauses

[[-90, 101],
 [-94, 102],
 [-31, 103],
 [5, 104],
 [-61, 105],
 [29, 106],
 [-98, 107],
 [-78, 108],
 [69, 109],
 [-12, 110],
 [-34, 111],
 [49, 112],
 [88, 113],
 [58, 114],
 [97, 115],
 [100, 116],
 [36, 117],
 [20, 118],
 [-17, 119],
 [-63, 120],
 [24, 121],
 [-48, 122],
 [60, 123],
 [59, 124],
 [11, 125],
 [32, 126],
 [-55, 127],
 [-93, 128]]

In [11]:
from pysat.card import CardEnc, EncType
for delta in [28, 25, 20, 18]:
    card_clauses = CardEnc.atmost(
        lits=list(delta_vars.values()),
        bound=delta,
        encoding=EncType.seqcounter,
        vpool=vpool
    )

    final_cnf = cnf.clauses + clauses + card_clauses.clauses

    f_prime = CNF()
    f_prime.nv = vpool.top 
    f_prime.clauses = final_cnf

    from pysat.solvers import Solver

    with Solver(name='kissat404', bootstrap_with=f_prime) as solver:
        sat = solver.solve()
        model = solver.get_model() if sat else None

    print(f'Sat: {sat}        DELTA: {delta}')

Sat: True        DELTA: 28
Sat: True        DELTA: 25
Sat: True        DELTA: 20
Sat: True        DELTA: 18


In [12]:
model_set = set(model)
all(any(lit in model_set for lit in clause) for clause in cnf.clauses)

True